# Unidad 2.1 - smolagents

Este notebook cubre los conceptos principales del framework **smolagents** de Hugging Face para construir agentes de IA.

**Temas cubiertos:**
1. CodeAgents — agentes que escriben y ejecutan código Python
2. ToolCallingAgents — agentes que usan JSON para llamar herramientas
3. Retrieval Agents — agentes con búsqueda web y base de conocimiento personalizada

> Alfred, el mayordomo de Wayne Manor, es nuestro agente protagonista en esta unidad. 🦇

## Instalación y configuración

In [1]:
%pip install "smolagents[litellm]" -U
%pip install ddgs
%pip install langchain-community langchain-text-splitters rank_bm25
%pip install ipywidgets
%pip install markdownify requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from huggingface_hub import login

# Inicia sesión en Hugging Face Hub para acceder a la Serverless Inference API.
# Si tienes el token como variable de entorno (HF_TOKEN) ya estás autenticado.
# Alternativa sin widget: login(token="hf_tu_token_aquí")
try:
    login()
except ImportError:
    # ipywidgets no disponible — usa token directamente
    import os
    token = os.environ.get("HF_TOKEN", "")
    if token:
        login(token=token)
    else:
        print("Instala ipywidgets o define la variable de entorno HF_TOKEN.")
        print("Ejemplo: set HF_TOKEN=hf_tu_token  (Windows)")
        print("         $env:HF_TOKEN='hf_tu_token'  (PowerShell)")

---
## Parte 1: CodeAgents

Los **CodeAgents** son el tipo de agente principal en `smolagents`. En lugar de generar JSON o texto, estos agentes producen **código Python** para realizar acciones.

**Ventajas del código sobre JSON:**
- **Composabilidad**: combinación y reutilización fácil de acciones
- **Gestión de objetos**: trabajo directo con estructuras complejas como imágenes
- **Generalidad**: permite expresar cualquier tarea computacionalmente posible
- **Natural para LLMs**: el código de alta calidad ya está presente en los datos de entrenamiento

### Ejemplo 1: Seleccionar una lista de reproducción para la fiesta

In [3]:
from smolagents import ToolCallingAgent, DuckDuckGoSearchTool, LiteLLMModel

model = LiteLLMModel(
    model_id="ollama_chat/qwen2:7b",
    api_base="http://127.0.0.1:11434",
    num_ctx=8192,
)

# ToolCallingAgent usa JSON para llamar herramientas; más compatible con modelos locales
agent = ToolCallingAgent(tools=[DuckDuckGoSearchTool()], model=model)

agent.run("Busca las mejores recomendaciones musicales para una fiesta en la mansion Wayne.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Busca las mejores recomendaciones musicales para una fiesta en la mansion Wayne.                                │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2:7b ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': 'mejores recomendaciones musicales para una fiesta en la   │
│ mansion Wayne'}                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: ## Search Results

|Recomendaciones de música y artistas para animar una 
fiesta](https://fiestasyartistas.com/imprescindibles-animar-cualquier-fiesta)
Un DJ especializado en reggaetón y música urbana es la mejor opción. Un buen DJ no solo conoce las últimas 
tendencias y los éxitos, sino que también sabe cómo mezclar y adaptar las canciones para mantener a todos en 
movimiento.

|Canciones que no pueden faltar en una fiesta - Descubre los temas musicales que nunca 
fallan](https://www.mundodeportivo.com/uncomo/fiesta/articulo/canciones-que-no-pueden-faltar-en-una-fiesta-50969.ht
ml)
June 30, 2022 - Más canciones para una fiesta - ¡grandes temas internacionales! ... Pop, rock, salsa, música 
electrónica, reggaetón o clásicos de los 80... en cada estilo musical tienes un sinfín de canciones perfectas para 
bailar en una fiesta.

|100 canciones para una fiesta de apartamento ideal](https://bacanika.com/articulo/100-canciones-para-una-fiesta)
Predominan los ritmos funkeros y el hip hop. Si la fiesta tiene muchos adultos de más de cincuenta años lo mejor es
empezar con Los Melódicos, la Billo’s Caracas Boys y algo de rock setentero.

|¿Que tipo de musica poner en una fiesta privada? Claves para acertar con el 
ambiente](https://www.deamare.es/post/que-musica-poner-para-una-fiesta)
June 17, 2025 - ¿Organizas una fiesta privada y no sabes qué música elegir? Te contamos que tipo de musica poner 
para una fiesta según el estilo del evento, el público y el ambiente que deseas crear.

|Música para una fiesta temática en casa - Bloygo - Yoigo](https://bloygo.yoigo.com/musica-para-fiesta-casa/)
May 10, 2022 - Si a ti lo que te gusta es ser el anfitrión perfecto y organizar encuentros para disfrutar con tus 
amigos, en este post te dejamos unas cuantas listas de Spotify con las mejores canciones para una fiesta temática 
en casa.

|TOP 100: La ‘playlist’ definitiva para tus 
fiestas](https://www.cosmopolitan.com/es/entretenimiento-cultura/a36927909/mejores-canciones-fiesta-playlist/)
August 27, 2022 - El término ‘fiesta’ viene del latín ‘feste’. En la RAE lo definen como ‘‘reunión de gente para 
celebrar algo o divertirse’’. Aunque nosotras preferimos catalogarla como ‘‘la que te va a dar esta’’. Oye, no nos 
malinterpretes, porque ‘esta’ es una ‘playlist’ que hemos preparado para ti con las mejores canciones que te puedas
imaginar para un evento y, además, cumpliendo con todos los requisitos necesarios para que se convierta en la 
definitiva. ¿Cuáles son estos? Una gran variedad de temas con estilos musicales diversos, hits actuales y guiños a 
la nostalgia (porque los 2000 fueron muy fuertes), y una larga duración para que no tengas que ir cambiando de 
lista cada poco tiempo, que sabemos que es un rollazo.

|¿Cómo elegir la playlist para fiesta perfecta? Consejos y 
trucos](https://salagingermadrid.com/como-elegir-la-playlist-para-fiesta-perfecta-consejos-y-trucos/)
October 30, 2025 - Consejos y trucos para elegir la mejor playlist de fiesta. ¡Lee nuestras recomendaciones y 
acierta seguro, no pararás de bailar!

|¿Qué tipo de música para eventos es la más adecuada?](https://elolivar.es/blog/musica-para-eventos/)
November 28, 2023 - Desde éxitos actuales hasta canciones exitosas de los últimos años, las opciones son infinitas.
Ten en cuenta las preferencias de tus invitados y personaliza la lista de reproducción según sus gustos. Al elegir 
la música adecuada, puedes garantizar que tu fiesta sea una experiencia inolvidable y agradable para todos los 
asistentes.

|30 Mejores Canciones para Animar una Fiesta: ¡Lista 
Definitiva!](https://malasombra.net/post/mejores-canciones-para-animar-una-fiesta.html)
March 7, 2025 - ¡Prepárate para mover el cuerpo! Encontrar la música perfecta para animar una fiesta puede ser un 
desafío, pero hemos seleccionado una lista definitiva de canciones que garantizan una noche inolvidable. Desde 
ritmos contagiosos que te harán bailar sin parar hasta melodías latinas que te transportarán a un a

[Step 1: Duration 138.56 seconds| Input tokens: 1,095 | Output tokens: 39]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': "Consider hiring a specialized DJ for reggaeton and     │
│ urban music. Prepare a playlist that includes a variety of genres like pop, rock, salsa, electronic music, and  │
│ reggaeton along with hits from the early 2000s to accommodate all tastes. Also, explore resources such as       │
│ Bacanika's list of 100 songs for a party or Cosmopolitan's definitive 'playlist' for your event."}              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Consider hiring a specialized DJ for reggaeton and urban music. Prepare a playlist that includes a 
variety of genres like pop, rock, salsa, electronic music, and reggaeton along with hits from the early 2000s to 
accommodate all tastes. Also, explore resources such as Bacanika's list of 100 songs for a party or Cosmopolitan's 
definitive 'playlist' for your event.

Final answer: Consider hiring a specialized DJ for reggaeton and urban music. Prepare a playlist that includes a 
variety of genres like pop, rock, salsa, electronic music, and reggaeton along with hits from the early 2000s to 
accommodate all tastes. Also, explore resources such as Bacanika's list of 100 songs for a party or Cosmopolitan's 
definitive 'playlist' for your event.

[Step 3: Duration 81.88 seconds| Input tokens: 6,443 | Output tokens: 945]

"Consider hiring a specialized DJ for reggaeton and urban music. Prepare a playlist that includes a variety of genres like pop, rock, salsa, electronic music, and reggaeton along with hits from the early 2000s to accommodate all tastes. Also, explore resources such as Bacanika's list of 100 songs for a party or Cosmopolitan's definitive 'playlist' for your event."

### Ejemplo 2: Herramienta personalizada para preparar el menú

Usamos el decorador `@tool` para definir una función personalizada que actúa como herramienta.

In [4]:
from smolagents import ToolCallingAgent, tool, LiteLLMModel

@tool
def suggest_menu(occasion: str) -> str:
    """
    Sugiere un menu basado en el tipo de ocasion para la fiesta de Alfred.
    Llama esta herramienta cuando necesites recomendar comida o bebidas para un evento.
    Por ejemplo: suggest_menu(occasion='formal') -> menu de 3 platos con vino.
    Args:
        occasion (str): El tipo de ocasion para la fiesta. Valores permitidos:
                        - "casual": Menu para fiesta informal.
                        - "formal": Menu para fiesta formal.
                        - "superhero": Menu para fiesta de superheroes.
                        - "custom": Menu personalizado.
    """
    if occasion == "casual":
        return "Pizza, snacks y bebidas."
    elif occasion == "formal":
        return "Cena de 3 platos con vino y postre."
    elif occasion == "superhero":
        return "Buffet con comida energetica y saludable."
    else:
        return "Menu personalizado para el mayordomo."

model = LiteLLMModel(
    model_id="ollama_chat/qwen2:7b",
    api_base="http://127.0.0.1:11434",
    num_ctx=8192,
)

# ToolCallingAgent usa JSON para llamar herramientas; más compatible con modelos locales
agent = ToolCallingAgent(tools=[suggest_menu], model=model)

agent.run("Prepara un menu formal para la fiesta.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Prepara un menu formal para la fiesta.                                                                          │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2:7b ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'suggest_menu' with arguments: {'occasion': 'formal'}                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Cena de 3 platos con vino y postre.

[Step 1: Duration 53.47 seconds| Input tokens: 1,260 | Output tokens: 26]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Cena de 3 platos con vino y postre'}                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Cena de 3 platos con vino y postre

Final answer: Cena de 3 platos con vino y postre

[Step 2: Duration 13.72 seconds| Input tokens: 2,590 | Output tokens: 61]

'Cena de 3 platos con vino y postre'

### Ejemplo 3: Herramientas para tareas de tiempo y cálculo

Con `ToolCallingAgent`, en lugar de importar módulos Python directamente en el código generado, creamos una **herramienta** que expone esa funcionalidad. Aquí definimos `get_current_time()` para que Alfred pueda calcular cuándo estará lista la fiesta.

> **Nota:** `CodeAgent` permite importaciones adicionales via `additional_authorized_imports`, pero requiere que el modelo genere código con un formato muy específico (`<code>...</code>`). `ToolCallingAgent` es más robusto con modelos locales porque usa llamadas a herramientas en formato JSON.

In [5]:
from smolagents import ToolCallingAgent, LiteLLMModel, tool
from datetime import datetime, timedelta

@tool
def get_current_time() -> str:
    """
    Obtiene la hora actual del sistema.
    Returns:
        La hora actual en formato HH:MM.
    """
    return datetime.now().strftime("%H:%M")

model = LiteLLMModel(
    model_id="ollama_chat/qwen2:7b",
    api_base="http://127.0.0.1:11434",
    num_ctx=8192,
)

agent = ToolCallingAgent(tools=[get_current_time], model=model)

agent.run(
    """
    Alfred necesita preparar la fiesta. Estas son las tareas:
    1. Preparar las bebidas - 30 minutos
    2. Decorar la mansion - 60 minutos
    3. Preparar el menu - 45 minutos
    4. Preparar la musica y la lista de reproduccion - 45 minutos

    Si comenzamos ahora mismo, a que hora estara lista la fiesta?
    """
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Alfred necesita preparar la fiesta. Estas son las tareas:                                                       │
│     1. Preparar las bebidas - 30 minutos                                                                        │
│     2. Decorar la mansion - 60 minutos                                                                          │
│     3. Preparar el menu - 45 minutos                                                                            │
│     4. Preparar la musica y la lista de reproduccion - 45 minutos                                               │
│                                                                                                                 │
│     Si comenzamos ahora mismo, a que hora estara lista la fiesta?                                               │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2:7b ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_current_time' with arguments: {}                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 16:43

[Step 1: Duration 41.22 seconds| Input tokens: 1,114 | Output tokens: 22]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': '22:27'}                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 22:27

Final answer: 22:27

[Step 5: Duration 12.48 seconds| Input tokens: 6,429 | Output tokens: 295]

'22:27'

### Ejemplo 4: Agente completo para planificar la fiesta

Combinamos múltiples herramientas para crear el agente "Alfred" definitivo.

In [6]:
from smolagents import (
    ToolCallingAgent, DuckDuckGoSearchTool, FinalAnswerTool,
    LiteLLMModel, Tool, tool, VisitWebpageTool
)

@tool
def suggest_menu(occasion: str) -> str:
    """
    Sugiere un menu basado en el tipo de ocasion para la fiesta de Alfred.
    Llama esta herramienta cuando necesites recomendar comida o bebidas para un evento.
    Por ejemplo: suggest_menu(occasion='formal') -> menu de 3 platos con vino.
    Args:
        occasion: El tipo de ocasion para la fiesta.
    """
    if occasion == "casual":
        return "Pizza, snacks y bebidas."
    elif occasion == "formal":
        return "Cena de 3 platos con vino y postre."
    elif occasion == "superhero":
        return "Buffet con comida energetica y saludable."
    else:
        return "Menu personalizado para el mayordomo."

@tool
def catering_service_tool(query: str) -> str:
    """
    Devuelve el servicio de catering mejor valorado en Ciudad Gotica.
    Llama esta herramienta cuando necesites encontrar o recomendar un servicio de catering.
    Por ejemplo: catering_service_tool(query='catering Gotham') -> nombre del mejor servicio.
    Args:
        query: Termino de busqueda para encontrar servicios de catering.
    """
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }
    best_service = max(services, key=services.get)
    return best_service

class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = (
        "Usa esta herramienta cuando necesites ideas creativas para el tema de una fiesta de superheroes. "
        "Por ejemplo: superhero_party_theme_generator(category='villain masquerade') -> idea para fiesta de villanos. "
        "Categorias disponibles: 'classic heroes', 'villain masquerade', 'futuristic Gotham'."
    )

    inputs = {
        "category": {
            "type": "string",
            "description": "El tipo de fiesta de superheroes (ej: 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }
    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Gala de la Liga de la Justicia: Los invitados vienen disfrazados de sus heroes favoritos de DC.",
            "villain masquerade": "Baile de mascaras de los Rogues de Gotham: Una mascarada misteriosa con villanos clasicos de Batman.",
            "futuristic gotham": "Neo-Gotham Night: Fiesta cyberpunk inspirada en Batman Beyond, con decoraciones de neon."
        }
        return themes.get(category.lower(), "Tema no encontrado. Prueba 'classic heroes', 'villain masquerade', o 'futuristic Gotham'.")

model = LiteLLMModel(
    model_id="ollama_chat/qwen2:7b",
    api_base="http://127.0.0.1:11434",
    num_ctx=8192,
)

# Alfred, el mayordomo, con todas sus herramientas
agent = ToolCallingAgent(
    tools=[
        DuckDuckGoSearchTool(),
        VisitWebpageTool(),
        suggest_menu,
        catering_service_tool,
        SuperheroPartyThemeTool(),
        FinalAnswerTool()
    ],
    model=model,
    max_steps=10,
    verbosity_level=2
)

agent.run("Dame la mejor lista de reproduccion para una fiesta en la mansion Wayne. El tema es una 'mascarada de villanos'.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Dame la mejor lista de reproduccion para una fiesta en la mansion Wayne. El tema es una 'mascarada de           │
│ villanos'.                                                                                                      │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2:7b ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Action:                                                                                                            
{                                                                                                                  
  "name": "superhero_party_theme_generator",                                                                       
  "arguments": {"category": "villain masquerade"}                                                                  
}                                                                                                                  

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'superhero_party_theme_generator' with arguments: {'category': 'villain masquerade'}              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Baile de mascaras de los Rogues de Gotham: Una mascarada misteriosa con villanos clasicos de Batman.

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Baile de mascaras de los Rogues de Gotham: Una         │
│ mascarada misteriosa con villanos clasicos de Batman. Menu personalizado para el mayordomo.'}                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Baile de mascaras de los Rogues de Gotham: Una mascarada misteriosa con villanos clasicos de Batman. 
Menu personalizado para el mayordomo.

Final answer: Baile de mascaras de los Rogues de Gotham: Una mascarada misteriosa con villanos clasicos de Batman. 
Menu personalizado para el mayordomo.

[Step 3: Duration 19.01 seconds| Input tokens: 6,102 | Output tokens: 122]

'Baile de mascaras de los Rogues de Gotham: Una mascarada misteriosa con villanos clasicos de Batman. Menu personalizado para el mayordomo.'

---
## Parte 2: ToolCallingAgents

Los **ToolCallingAgents** son el segundo tipo de agente en `smolagents`. A diferencia de los CodeAgents que usan Python, estos agentes usan las **capacidades nativas de llamada a herramientas** de los proveedores de LLM para generar estructuras **JSON**.

### Comparación: CodeAgent vs ToolCallingAgent

**CodeAgent** genera y ejecuta código Python:
```python
for query in ["Mejores servicios de catering en Gotham", "Ideas para fiestas de superhéroes"]:
    print(web_search(f"Búsqueda: {query}"))
```

**ToolCallingAgent** genera una estructura JSON:
```python
[
    {"name": "web_search", "arguments": "Mejores servicios de catering en Gotham"},
    {"name": "web_search", "arguments": "Ideas para fiestas de superhéroes"}
]
```

In [3]:
from smolagents import ToolCallingAgent, DuckDuckGoSearchTool, LiteLLMModel

model = LiteLLMModel(
    model_id="ollama_chat/qwen2:7b",
    api_base="http://127.0.0.1:11434",
    num_ctx=8192,
)

# ToolCallingAgent genera estructuras JSON (no código Python) usando las capacidades
# nativas de tool-calling del modelo — más robusto con modelos locales como qwen2:7b
agent = ToolCallingAgent(
    tools=[DuckDuckGoSearchTool()],
    model=model
)

result = agent.run("Busca las mejores recomendaciones musicales para una fiesta en la mansion Wayne.")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Busca las mejores recomendaciones musicales para una fiesta en la mansion Wayne.                                │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2:7b ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': 'Mejores recomendaciones musicales para una fiesta en la   │
│ mansión Wayne'}                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: ## Search Results

|Hablamos de... 'Érase una vez en... Hollywood' 
de](http://mevadecine.com/hablamos-de-erase-una-vez-en-hollywood-de-quentin-tarantino/)
El humor y la connivencia del espectador resultan claves para sostener una cinta que se acerca a las tres horas y 
no posee un desarrollo, ni una ...

|Jaime Molina nos presenta las mil caras de una historia en 
su](http://www.pandora-magazine.com/literatura/jaime-molina-nos-presenta-las-mil-caras-de-una-historia-en-su-nueva-
novela-una-casa-respetable/)
El escritor Jaime Molina utiliza una “estructura poliédrica y caleidoscópica”, para dotar a la narración de su 
novela de un innovador punto de ...

|Las 100 mejores canciones para una fiesta (con 
'playlist')](https://www.cosmopolitan.com/es/entretenimiento-cultura/a36927909/mejores-canciones-fiesta-playlist/)
Las 100 mejores canciones para una fiesta (con 'playlist') La 'playlist' definitiva a la que recurrir cuando no 
sabes qué poner durante una velada con amigos. ¡Empieza la 'party'!

|Las 200 mejores playlists de Spotify para cada momento y estilo 
musical](https://ordago13.substack.com/p/todas-mis-playlists-de-spotify)
eo-soul, blues, folk, country, grunge, metal, punk y más. Desde clásicos imprescindibles hasta nuevas tendencias, 
cada lista está diseñada para ofrecerte una experiencia musical única. Música en español: las mejores playlists Si 
prefieres la música en español, aquí tienes las listas más completas: desde la mejor playlist de Indie Español 
hasta los grandes éxitos del rock en ...

|≫ Mejores Listas De Spotify Para FIESTAS 
2026](https://elpoderdelandroideverde.com/listas-de-spotify-para-fiestas/)
¿Necesitas celebrar una FIESTA y no sabes que MÚSICA es la MÁS ADECUADAS ?. Entra aquí ☝ y descubre las ⭐ MEJORES 
LISTAS de SPOTIFY para FIESTAS ⭐.

|Cómo Seleccionar la Música Perfecta para Cada 
Evento](https://soundsmarket.com/blog/como-seleccionar-la-musica-perfecta-para-cada-evento)
Tarde con amigos: Opta por una playlist variada con diferentes géneros, desde clásicos hasta éxitos recientes. 
Noche romántica: Una playlist de baladas y canciones de amor puede proporcionar un ambiente íntimo y agradable. 
Barbacoa o fiesta en la piscina: Elige canciones veraniegas, pop o relajantes para un evento al aire libre.

|¿Cómo crear la playlist perfecta para tu fiesta? - 
Weezevent](https://weezevent.com/es/blog/crear-playlist-fiesta/)
¡Vamos a responder a todas estas preguntas! Aquí tienes una guía rápida para ayudarte a crear la playlist perfecta 
para tu fiesta, teniendo en cuenta factores como el tema de la fiesta, tus invitados y vuestros gustos musicales. 
También encontrarás una lista de ideas originales de canciones ¡que querrás añadir directamente a tu playlist!

|Música Perfecta: Claves para Eventos Inolvidables](https://kaboomeventos.com.ar/musica-de-fondo-para-eventos/)
¿Organizando un evento y no sabes qué música poner? Descubre la guía definitiva para elegir la banda sonora ideal 
que hará vibrar a tus invitados. ¡Convierte tu evento en una fiesta memorable!

|Las 30 Mejores Canciones para Fiestas: ¡Lista 
Definitiva!](https://malasombra.net/post/mejores-canciones-de-fiesta.html)
¡Descubre nuestra lista definitiva con las 30 mejores canciones para animar cualquier fiesta! Desde éxitos clásicos
hasta temas modernos, te ofrecemos una selección musical que pondrá a bailar a todos.

|Lista de canciones de fiesta | fundacionlotus](https://fundacionlotus.cl/articulos/lista-de-canciones-de-fiesta/)
En definitiva, crear la lista de canciones perfecta para una fiesta es un proceso que requiere planificación, 
consideración y flexibilidad. Siguiendo estos consejos, podrás confeccionar una lista de reproducción que mantendrá
a tus invitados bailando y disfrutando durante toda la noche.

[Step 1: Duration 24.90 seconds| Input tokens: 1,095 | Output tokens: 38]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': "Recomendaciones de canciones para tu fiesta en la      │
│ mansión Wayne: Celebration de Kool & The Gang, Can't Stop the Feeling! de Justin Timberlake, Uptown Funk de     │
│ Mark Ronson ft. Bruno Mars, Get Lucky de Daft Punk ft. Pharrell Williams, Happy de Pharrell Williams, The Twist │
│ de Chubby Checker, You Can't Stop the Feeling! de Justin Timberlake, Get Down Tonight de Flo Rida ft. Sia, I    │
│ Want You Back de The Jackson 5 y Dancing Queen de ABBA."}                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Recomendaciones de canciones para tu fiesta en la mansión Wayne: Celebration de Kool & The Gang, 
Can't Stop the Feeling! de Justin Timberlake, Uptown Funk de Mark Ronson ft. Bruno Mars, Get Lucky de Daft Punk ft.
Pharrell Williams, Happy de Pharrell Williams, The Twist de Chubby Checker, You Can't Stop the Feeling! de Justin 
Timberlake, Get Down Tonight de Flo Rida ft. Sia, I Want You Back de The Jackson 5 y Dancing Queen de ABBA.

Final answer: Recomendaciones de canciones para tu fiesta en la mansión Wayne: Celebration de Kool & The Gang, 
Can't Stop the Feeling! de Justin Timberlake, Uptown Funk de Mark Ronson ft. Bruno Mars, Get Lucky de Daft Punk ft.
Pharrell Williams, Happy de Pharrell Williams, The Twist de Chubby Checker, You Can't Stop the Feeling! de Justin 
Timberlake, Get Down Tonight de Flo Rida ft. Sia, I Want You Back de The Jackson 5 y Dancing Queen de ABBA.

[Step 2: Duration 249.38 seconds| Input tokens: 3,374 | Output tokens: 565]

Recomendaciones de canciones para tu fiesta en la mansión Wayne: Celebration de Kool & The Gang, Can't Stop the Feeling! de Justin Timberlake, Uptown Funk de Mark Ronson ft. Bruno Mars, Get Lucky de Daft Punk ft. Pharrell Williams, Happy de Pharrell Williams, The Twist de Chubby Checker, You Can't Stop the Feeling! de Justin Timberlake, Get Down Tonight de Flo Rida ft. Sia, I Want You Back de The Jackson 5 y Dancing Queen de ABBA.


**Nota:** En la traza del agente verás `Calling tool: 'web_search' with arguments: {...}` en lugar de `Executing parsed code:`, que es la diferencia principal entre ambos tipos.

---
## Parte 3: Retrieval Agents (RAG Agéntico)

Los sistemas **RAG (Retrieval-Augmented Generation)** combinan recuperación de información con generación para dar respuestas conscientes del contexto.

El RAG agéntico extiende esto permitiendo al agente:
- Formular consultas de búsqueda autónomamente
- Criticar los resultados recuperados
- Realizar múltiples pasos de recuperación

### Ejemplo 1: Recuperación básica con DuckDuckGo

In [4]:
from smolagents import ToolCallingAgent, DuckDuckGoSearchTool, LiteLLMModel

search_tool = DuckDuckGoSearchTool()
model = LiteLLMModel(
    model_id="ollama_chat/qwen2:7b",
    api_base="http://127.0.0.1:11434",
    num_ctx=8192,
)

agent = ToolCallingAgent(
    model=model,
    tools=[search_tool],
)

response = agent.run(
    "Busca ideas de fiesta de superheroes de lujo, incluyendo decoraciones, entretenimiento y catering."
)
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Busca ideas de fiesta de superheroes de lujo, incluyendo decoraciones, entretenimiento y catering.              │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2:7b ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': 'ideas para fiestas de superhéroes de lujo con decoración, │
│ entretenimiento y catering'}                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: ## Search Results

|Todo Eventos y Decoración Paraguay - Facebook](https://www.facebook.com/groups/1552152201693750/)
Decoraciones para todo tipo de eventos ¡Haz que tu evento se vea increíble! ✨ ✨ Decoraciones de lujo para tus 
eventos ✨ ¿Quieres que tu fiesta se vea así de ...

|4 Year Old Girl Bedroom Ideas and Decor - 
TikTok](https://www.tiktok.com/@louise_andallthingshome/video/7025534863025474822)
Nov 1, 2021 · 90s Room Decor Men · Decoracion De Cuarto Para Niñas · Small Bedroom Design for Two Boys. 4 Year Old 
Girl Bedroom Ideas and Decor. Discover cute ...

|EVENTOS_MOLONES | Facebook](https://www.facebook.com/groups/205397060814932/)
Decoraciones para todo tipo de eventos ¡Haz que tu evento se vea increíble! ✨ ✨ Decoraciones de lujo para tus 
eventos ✨ ¿Quieres que tu fiesta se vea así de ...

|Food Color Red Theme Party Table - TikTok](https://www.tiktok.com/discover/food-color-red-theme-party-table)
7 days ago · candy bar de Spiderman para cumpleaños, ideas de decoración temática para fiestas, mesa de chuches 
roja y azul, cumpleaños de superhéroes ...

|Dulces dieciséis: Looks de Fiesta en Roblox - TikTok](https://www.tiktok.com/@ximenadti/video/7502210831342472456)
May 8, 2025 · 895 me gusta,21 comentarios.Video de TikTok de ✨Ximena✨ (@ximenadti): “Descubre los estilos más 
tiernos para un cumpleaños dulce 16, ...

|Eclipse Kids (@eclipsekidseventos) · Villa Ballester - 
Instagram](https://www.instagram.com/eclipsekidseventos/?hl=en)
¡Bienvenidos a nuestro salón de eventos infantiles! Descubre cada rincón de nuestro espacio diseñado para hacer que
los sueños de los niños cobren vida.Missing: ideas | Show results with:ideas

|DIY Coraline Cake Gift Box Tutorial | Handmade Crafts - 
TikTok](https://www.tiktok.com/@thestayathomeaunt/video/7430895260248689963)
Oct 28, 2024 · Ideas De Decoración Para Una Bienvenida · Front Page Decoration Ideas ... catering para fiestas, 
pastelería en redes sociales, D'Liz repostería ...

|Doral Chamber of Commerce Calendar of Events in Doral, Miami ...](https://www.doralchamber.org/events/)
Trucos y consejos para crear decoraciones únicas. Todo el material necesario (flores premium, bases de lujo, y 
más). Certificado de participación para cada ...

|Power Birthday Party Theme - TikTok](https://www.tiktok.com/discover/power-birthday-party-theme)
May 11, 2026 · Celebra un cumpleaños inolvidable con un tema de Power Rangers. Descubre ideas creativas y 
decoraciones divertidas para toda la familia.

|Create your dream home now @Planner5d](https://planner5d.com/sitemap.html)
Main. Get ideas · Upload a plan · Design School · Design battle · Use Cases · About Us · Pricing · Gift Card · 
Newsroom · Design Council · Legal Hub ...

[Step 1: Duration 102.94 seconds| Input tokens: 1,102 | Output tokens: 46]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': ["Para una fiesta de superhéroes de lujo, necesitarás   │
│ decoraciones de alta calidad disponibles en todo evento y decoración Paraguay. El entretenimiento puede incluir │
│ actividades temáticas como un 'candy bar' de Spiderman y un cumpleaños dulce 16. También puedes considerar      │
│ contratar una compañía especializada en catering para fiestas o repostería como D'Liz repostería."]}            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: |"Para una fiesta de superhéroes de lujo, necesitarás decoraciones de alta calidad disponibles en 
todo evento y decoración Paraguay. El entretenimiento puede incluir actividades temáticas como un 'candy bar' de 
Spiderman y un cumpleaños dulce 16. También puedes considerar contratar una compañía especializada en catering para
fiestas o repostería como D'Liz repostería."]

Final answer: ["Para una fiesta de superhéroes de lujo, necesitarás decoraciones de alta calidad disponibles en 
todo evento y decoración Paraguay. El entretenimiento puede incluir actividades temáticas como un 'candy bar' de 
Spiderman y un cumpleaños dulce 16. También puedes considerar contratar una compañía especializada en catering para
fiestas o repostería como D'Liz repostería."]

[Step 2: Duration 118.63 seconds| Input tokens: 3,096 | Output tokens: 166]

["Para una fiesta de superhéroes de lujo, necesitarás decoraciones de alta calidad disponibles en todo evento y decoración Paraguay. El entretenimiento puede incluir actividades temáticas como un 'candy bar' de Spiderman y un cumpleaños dulce 16. También puedes considerar contratar una compañía especializada en catering para fiestas o repostería como D'Liz repostería."]


### Ejemplo 2: Herramienta de base de conocimiento personalizada

Para tareas especializadas, una base de conocimiento personalizada es invaluable. Usamos un recuperador **BM25** para búsqueda semántica sobre documentos propios.

In [5]:
from langchain_community.docstore.document import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from smolagents import Tool
from langchain_community.retrievers import BM25Retriever
from smolagents import ToolCallingAgent, LiteLLMModel

class PartyPlanningRetrieverTool(Tool):
    name = "party_planning_retriever"
    description = (
        "Usa esta herramienta cuando necesites ideas de planificacion de fiestas para la fiesta de superheroes de Alfred. "
        "Por ejemplo: party_planning_retriever(query='decoracion villanos') -> ideas de decoracion oscura. "
        "Busca en la base de conocimiento interna antes de usar busqueda web."
    )
    inputs = {
        "query": {
            "type": "string",
            "description": "La consulta a realizar. Debe estar relacionada con planificacion de fiestas o tematicas de superheroes.",
        }
    }
    output_type = "string"

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)
        self.retriever = BM25Retriever.from_documents(
            docs, k=5
        )

    def forward(self, query: str) -> str:
        assert isinstance(query, str), "La consulta debe ser una cadena de texto"
        docs = self.retriever.invoke(query)
        return "\nIdeas recuperadas:\n" + "".join(
            [
                f"\n\n===== Idea {str(i)} =====\n" + doc.page_content
                for i, doc in enumerate(docs)
            ]
        )

party_ideas = [
    {"text": "Un baile de mascaras con tematica de superheroes con decoracion de lujo, incluyendo acentos dorados y cortinas de terciopelo.", "source": "Ideas de Fiesta 1"},
    {"text": "Contratar un DJ profesional que pueda poner musica tematica de superheroes como Batman y Wonder Woman.", "source": "Ideas de Entretenimiento"},
    {"text": "Para el catering, sirve platos con nombre de superheroes, como 'El Batido Verde del Hulk' y 'El Filete de Poder de Iron Man'.", "source": "Ideas de Catering"},
    {"text": "Decora con los logos iconicos de superheroes y proyecciones de Gotham y otras ciudades de superheroes alrededor del venue.", "source": "Ideas de Decoracion"},
    {"text": "Experiencias interactivas con VR donde los invitados puedan participar en simulaciones de superheroes o competir en juegos tematicos.", "source": "Ideas de Entretenimiento"}
]

source_docs = [
    Document(page_content=doc["text"], metadata={"source": doc["source"]})
    for doc in party_ideas
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True,
    strip_whitespace=True,
    separators=["\n\n", "\n", ".", " ", ""],
)
docs_processed = text_splitter.split_documents(source_docs)

party_planning_retriever = PartyPlanningRetrieverTool(docs_processed)

model = LiteLLMModel(
    model_id="ollama_chat/qwen2:7b",
    api_base="http://127.0.0.1:11434",
    num_ctx=8192,
)

agent = ToolCallingAgent(tools=[party_planning_retriever], model=model)

response = agent.run(
    "Encuentra ideas para una fiesta de superheroes de lujo, incluyendo opciones de entretenimiento, catering y decoracion."
)

print(response)

C:\Users\USUARIO\AppData\Roaming\Python\Python314\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Encuentra ideas para una fiesta de superheroes de lujo, incluyendo opciones de entretenimiento, catering y      │
│ decoracion.                                                                                                     │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2:7b ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'party_planning_retriever' with arguments: {'query': 'ideas for a luxury superhero birthday party │
│ including entertainment, catering and decoration'}                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Ideas recuperadas:


===== Idea 0 =====
Experiencias interactivas con VR donde los invitados puedan participar en simulaciones de superheroes o competir en
juegos tematicos.

===== Idea 1 =====
Decora con los logos iconicos de superheroes y proyecciones de Gotham y otras ciudades de superheroes alrededor del
venue.

===== Idea 2 =====
Para el catering, sirve platos con nombre de superheroes, como 'El Batido Verde del Hulk' y 'El Filete de Poder de 
Iron Man'.

===== Idea 3 =====
Contratar un DJ profesional que pueda poner musica tematica de superheroes como Batman y Wonder Woman.

===== Idea 4 =====
Un baile de mascaras con tematica de superheroes con decoracion de lujo, incluyendo acentos dorados y cortinas de 
terciopelo.

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': "Las ideas para una fiesta de superheroes de lujo       │
│ incluyen experiencias interactivas con VR, decoraciones temáticas que destacan los logotipos y escenas de       │
│ películas, catering con platos inspirados en personajes de superhéroes como 'El Batido Verde del Hulk',         │
│ contratación de un DJ para reproducir música temática de superheroes, y un baile de máscaras temático de        │
│ superheroínas y superhombres. Todo esto se complementa con una decoración de lujo que incluye dorado y cortinas │
│ de terciopelo."}                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Las ideas para una fiesta de superheroes de lujo incluyen experiencias interactivas con VR, 
decoraciones temáticas que destacan los logotipos y escenas de películas, catering con platos inspirados en 
personajes de superhéroes como 'El Batido Verde del Hulk', contratación de un DJ para reproducir música temática de
superheroes, y un baile de máscaras temático de superheroínas y superhombres. Todo esto se complementa con una 
decoración de lujo que incluye dorado y cortinas de terciopelo.

Final answer: Las ideas para una fiesta de superheroes de lujo incluyen experiencias interactivas con VR, 
decoraciones temáticas que destacan los logotipos y escenas de películas, catering con platos inspirados en 
personajes de superhéroes como 'El Batido Verde del Hulk', contratación de un DJ para reproducir música temática de
superheroes, y un baile de máscaras temático de superheroínas y superhombres. Todo esto se complementa con una 
decoración de lujo que incluye dorado y cortinas de terciopelo.

[Step 3: Duration 78.87 seconds| Input tokens: 4,647 | Output tokens: 236]

Las ideas para una fiesta de superheroes de lujo incluyen experiencias interactivas con VR, decoraciones temáticas que destacan los logotipos y escenas de películas, catering con platos inspirados en personajes de superhéroes como 'El Batido Verde del Hulk', contratación de un DJ para reproducir música temática de superheroes, y un baile de máscaras temático de superheroínas y superhombres. Todo esto se complementa con una decoración de lujo que incluye dorado y cortinas de terciopelo.


---
## Resumen

| Concepto | Descripción |
|---|---|
| **CodeAgent** | Genera y ejecuta código Python para realizar acciones |
| **ToolCallingAgent** | Genera estructuras JSON para llamar herramientas |
| **@tool decorator** | Forma simple de crear herramientas desde funciones Python |
| **Tool class** | Forma avanzada de crear herramientas con control total |
| **RAG agéntico** | El agente controla el proceso de recuperación y generación |
| **additional_authorized_imports** | Permite importar módulos adicionales en el sandbox |

**Recursos:**
- [Documentación oficial de smolagents](https://huggingface.co/docs/smolagents)
- [Ejecutable Code Actions Elicit Better LLM Agents](https://huggingface.co/papers/2402.01030)
- [Guía de RAG agéntico](https://huggingface.co/learn/cookbook/agent_rag)